# 01 — Basismodell am Rumpf

Ersetzt die frueheren Notebooks 01 (Basismodelle) und 04 (Achsen-Ablation).
Ergebnisse landen wie bisher je Teil in einem eigenen Ordner unter `results/`.

| Teil | Frage | Bericht |
|---|---|---|
| A | Schwellwert vs. Random Forest vs. 1D-CNN auf SisFall | Ergebnisse, Basismodell |
| B | Braucht man das Gyroskop? (3 vs. 6 Achsen) | Ergebnisse, Sensorik |

Legt `results/fall_model.keras` ab; alle weiteren Notebooks bauen darauf auf.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Ohne die
# aktuelle fall_lib waeren alle Ergebnisse falsch (andere Fensterlaenge).
import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("new_run", "build_windows", "znorm", "load_cappimu", "load_pilot",
           "augment_orientation", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
assert fl.WIN_LEN == 100, "falsche Fensterlaenge -- fall_lib pruefen"
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/ergebnisse"
MODEL = f"{BASE}/ergebnisse/fall_model.keras"        # Basismodell (Teil 01)
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)


def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte
    (Drive ist fuer tausende Kleindateien zu langsam). Gibt None
    zurueck, wenn der Ordner fehlt."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel


def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r


from tensorflow import keras
def lade_modell(pfad):
    if not Path(pfad).exists():
        raise FileNotFoundError(f"{pfad} fehlt -- vorheriges Notebook ausfuehren.")
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Konfiguration:", fl.default_config())

# Teil A — Basismodelle auf SisFall

## Daten laden

In [ ]:
SIS = hole("SisFall_dataset", "/content/SisFall")
trials = fl.load_sisfall(SIS)

# X_raw bleibt unnormalisiert (fuer Schwellwert und Random Forest)
X_raw, y, groups, acts = fl.build_windows(trials, mode="standard")
print(f"\nFenster: {X_raw.shape}, Stuerze: {int(y.sum())}")

run = fl.new_run("01_basismodelle")
tr, te = fl.subject_split(groups, y, test_size=0.25)
print(f"Train {len(tr)} ({len(np.unique(groups[tr]))} Prob.) | "
      f"Test {len(te)} ({len(np.unique(groups[te]))} Prob.)")
ergebnisse = {}

## 1 — Schwellwertverfahren

Ein Fenster gilt als Sturz, wenn der Betrag der Beschleunigung einen
festen Schwellwert ueberschreitet. Der Schwellwert wird auf den
Trainingsdaten bestimmt (Youden-Index) und anschliessend unveraendert
auf den Testsatz angewendet.

In [ ]:
best_thr, sweep = fl.threshold_sweep(X_raw[tr], y[tr])

print("\nAnwendung auf den Testsatz:")
pred = fl.threshold_predict(X_raw[te], acc_threshold=best_thr)
ergebnisse["threshold"] = fl.evaluate_predictions(
    y[te], pred, f"Schwellwert ({best_thr:.1f} g)")
ergebnisse["threshold"]["threshold_g"] = best_thr
ergebnisse["threshold_sweep"] = sweep

## 2 — Random Forest

18 Merkmale, je neun aus Beschleunigung und Drehrate. Alle beruhen
auf dem Vektorbetrag und sind damit **rotationsinvariant**: Die
Einbaulage des Sensors in der Sohle beeinflusst sie nicht — fuer eine
Einlegesohle wesentlich, da die Ausrichtung nicht garantiert werden
kann.

In [ ]:
F = fl.extract_features(X_raw)
print(f"Merkmalsmatrix: {F.shape}\n")

rf = fl.train_random_forest(F[tr], y[tr], n_estimators=200)
ergebnisse["random_forest"] = fl.evaluate_predictions(
    y[te], rf.predict(F[te]), "Random Forest")

### Merkmalswichtigkeit

Der Random Forest ist im Gegensatz zum CNN interpretierbar. Die
Verteilung der Wichtigkeit auf Beschleunigungs- und
Gyroskopmerkmale liefert einen empirischen Beitrag zur
Sensorempfehlung aus Kapitel 2.

In [ ]:
wichtig = fl.feature_importance(rf, top=10)
ergebnisse["feature_importance"] = wichtig

gyr_anteil = sum(r["importance"] for r in wichtig
                 if r["sensor"] == "Gyroskop")
print(f"\n  Gesamtanteil der Gyroskopmerkmale: {gyr_anteil:.1%}")
ergebnisse["gyro_importance_share"] = round(float(gyr_anteil), 4)

### Kontrolle der Rotationsinvarianz

Die Merkmale werden nach einer zufaelligen dreidimensionalen Drehung
erneut berechnet. Bleiben sie unveraendert, ist die Invarianz
bestaetigt.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.random(random_state=0).as_matrix().astype(np.float32)
X_rot = X_raw.copy()
X_rot[:, :, 0:3] = X_raw[:, :, 0:3] @ R.T
X_rot[:, :, 3:6] = X_raw[:, :, 3:6] @ R.T

F_rot = fl.extract_features(X_rot)
gleich = (rf.predict(F[te]) == rf.predict(F_rot[te])).mean()
print(f"  Groesste absolute Merkmalsabweichung: "
      f"{np.abs(F - F_rot).max():.2e}")
print(f"  Unveraenderte Vorhersagen: {gleich:.2%}")
ergebnisse["rotation_invariance"] = {
    "unchanged_predictions": round(float(gleich), 4)}

## 3 — 1D-CNN

Lernt die Merkmale selbst aus dem Rohsignal (100 × 6). Hier werden
z-normalisierte Eingaben verwendet.

In [ ]:
X = fl.znorm(X_raw)

model = fl.build_cnn(n_channels=6)
model.summary()
fl.train(model, X[tr], y[tr], epochs=30, verbose=1)

In [ ]:
print("\nSisFall — subjektunabhaengiger Testsatz\n")
ergebnisse["cnn"] = fl.evaluate(model, X[te], y[te], "1D-CNN")
ergebnisse["cnn"]["params"] = int(model.count_params())

model.save("/content/fall_model.keras")
shutil.copy("/content/fall_model.keras", MODEL)
shutil.copy("/content/fall_model.keras", run / "fall_model.keras")
print(f"\nModell abgelegt unter {MODEL}")

## Vergleich der drei Verfahren

In [ ]:
print(f"{'Verfahren':<24} {'Se':>8} {'Sp':>8} {'Acc':>8} "
      f"{'FP':>7} {'FN':>6}")
print("-"*66)
for key, name in [("threshold", "Schwellwert (Bourke)"),
                  ("random_forest", "Random Forest"),
                  ("cnn", "1D-CNN")]:
    r = ergebnisse[key]
    print(f"{name:<24} {r['Se']:>8.3f} {r['Sp']:>8.3f} "
          f"{r['Acc']:>8.3f} {r['FP']:>7d} {r['FN']:>6d}")

## int8-TFLite-Export

Fuer die Portierung auf den ESP32-C6.

In [ ]:
kb = fl.export_tflite_int8(model, X[tr],
                          run / "fall_model_int8.tflite")
ergebnisse["tflite_kb"] = round(kb, 1)

fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Schwellwert und Random Forest auf rohen Fenstern, "
           "CNN auf z-normalisierten. Identischer Testsatz."))

# Teil B — Achsen-Ablation: 3 gegen 6 Achsen

Drei Modelle, identische Architektur und Aufteilung; einziger Unterschied sind die Eingangskanaele.

In [ ]:
run = fl.new_run("04_achsen")
ergebnisse = {}

# Position waehlen: SisFall (Huefte) oder UMAFall ANKLE (Knoechel)
# Der Knoechel ist fuer die Insole aussagekraeftiger, da dort die
# Beschleunigung durch Fersenauftritte gesaettigt ist.
DATENSATZ = "sisfall"      # "sisfall" oder "umafall_ankle"

if DATENSATZ == "sisfall":
    trials = fl.load_sisfall(hole("SisFall_dataset", "/content/SisFall"))
else:
    trials = fl.load_umafall(
        hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
             "/content/UMAFall"), position="ANKLE")

X, y, g, _ = fl.build_windows(trials, mode="standard")
tr, te = fl.subject_split(g, y, test_size=0.25)
print(f"\nTrain {len(tr)} | Test {len(te)} Fenster")

In [ ]:
varianten = {"acc6": "6 Achsen (Acc+Gyr)",
             "acc3": "3 Achsen (nur Acc)",
             "gyr3": "3 Achsen (nur Gyr)"}
modelle, abl = {}, {}

for mode, name in varianten.items():
    Xn = fl.znorm(fl.select_channels(X, mode))
    m = fl.build_cnn(n_channels=Xn.shape[2])
    fl.train(m, Xn[tr], y[tr], epochs=30)
    print(f"\n{name}  ({m.count_params():,} Parameter)")
    r = fl.evaluate(m, Xn[te], y[te], "  ")
    r["params"] = int(m.count_params())
    abl[mode] = r; modelle[mode] = m

## Ergebnis

In [ ]:
print(f"{'Variante':<24} {'Se':>7} {'Sp':>7} {'Acc':>7} {'Param':>9}")
print("-"*58)
for k, name in varianten.items():
    r = abl[k]
    print(f"{name:<24} {r['Se']:>7.3f} {r['Sp']:>7.3f} "
          f"{r['Acc']:>7.3f} {r['params']:>9,}")

d = abl["acc6"]["Se"] - abl["acc3"]["Se"]
print(f"\nBeitrag des Gyroskops: {d*100:+.2f} Prozentpunkte "
      f"Sensitivitaet")
print(f"Differenz der Parameterzahl: "
      f"{abl['acc6']['params']-abl['acc3']['params']} "
      f"(nur die Eingangsschicht)")

if abl["acc6"]["Se"] > 0.97:
    print("\nHinweis: Beide Varianten liegen nahe der Obergrenze. "
          "Ein Deckeneffekt kann kleine Unterschiede verdecken; "
          "die Aussage gilt zunaechst nur fuer diese Position.")
ergebnisse["ablation"] = abl
ergebnisse["delta_se"] = round(float(d), 4)
ergebnisse["datensatz"] = DATENSATZ

## Externe Gegenprobe: UniMiB-SHAR

UniMiB-SHAR enthaelt ausschliesslich Beschleunigungsdaten und eignet
sich daher zur unabhaengigen Pruefung des 3-Achsen-Modells.

Zu beachten: Die Daten stammen von einem Smartphone mit einem
Messbereich von nur ±2 g; Sturzspitzen werden abgeschnitten.

In [ ]:
UNI = hole("UniMiB-SHAR/data", "/content/UniMiB")
if UNI:
    Xu, yu, gu = fl.load_unimib(UNI)
    Xu = fl.znorm(Xu)
    print()
    zs = fl.evaluate(modelle["acc3"], Xu, yu, "Zero-Shot")
    tru, teu = fl.subject_split(gu, yu, test_size=0.3)
    m_ft = fl.finetune(modelle["acc3"], Xu[tru], yu[tru])
    ft = fl.evaluate(m_ft, Xu[teu], yu[teu], "Fine-Tuned")
    ergebnisse["unimib"] = {"zero_shot": zs, "fine_tuned": ft}

fl.save_results(run, ergebnisse, config=fl.default_config())